<a href="https://colab.research.google.com/github/lautarodibartolo-ae/t8001-pre-procesado-de-datos/blob/main/clase-4-trabajo-final/04_trabajo_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>


# Clase 4 — El ejemplo del trabajo final

**Taller T8001 · Pre-procesado de datos**

Este notebook **no es una plantilla ni hay que completarlo**. Es un trabajo final en miniatura,
terminado, sobre un caso inventado de veinticinco filas. Está para que veas qué forma tiene una
entrega antes de hacer la tuya: las cinco piezas que pide la consigna aparecen acá resueltas, en
el orden en que se trabajan: la ficha primero, el diccionario después de la primera mirada, la
limpieza en el medio, y los gráficos, la exportación y el informe al final.

La consigna, la guía de evaluación y los datasets están en el documento de la clase:
[**T8001_Consigna_Trabajo_Final.pdf**](https://github.com/lautarodibartolo-ae/t8001-pre-procesado-de-datos/blob/main/clase-4-trabajo-final/T8001_Consigna_Trabajo_Final.pdf).

El caso es chico a propósito, para que cada decisión se pueda verificar mirando la tabla. Tu
dataset real va a tener cientos o miles de filas, y más problemas: el trabajo es el mismo, solo
que las cuentas las va a hacer pandas en lugar de tus ojos.

Los tres números de este caso, que vas a ver aparecer paso a paso:

> Veinticinco filas en el archivo. Veinticuatro préstamos al sacar el duplicado. Veintiún estados
> de devolución con dato.


---
# La ficha de la fuente

En un trabajo real, casi todo esto sale de la ficha del portal y de lo que sepas del organismo.
Acá el dato es inventado, así que la ficha es corta y declara sus supuestos.

- **Quién lo produce:** la red de bibliotecas populares de un municipio, con tres sedes. Usa el
  dato para gestionar los préstamos, no para publicarlo.
- **Mecanismo de captura:** carga manual en una planilla, en la mesa de préstamo de cada sede. Es
  uno de los siete mecanismos de la clase 2, y comparte con el formulario el defecto del texto
  libre: la misma categoría escrita de varias formas, que es exactamente lo que vamos a encontrar
  en `sede`.
- **Sesgo esperable:** mide préstamos registrados, no lectura. El libro que se consulta en la
  sala y no se anota, no existe.
- **Unidad de observación:** un préstamo. No un libro ni un socio: el mismo libro prestado dos
  veces son dos filas.
- **Cobertura:** junio de 2025, las tres sedes.
- **Formato:** CSV, separado por comas, encoding UTF-8.
- **Licencia y frecuencia de actualización:** no informado. En un dataset real esto se busca en
  el portal, y si no está, decir que no está también es un hallazgo.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

Path("crudo").mkdir(exist_ok=True)
Path("generado").mkdir(exist_ok=True)

tabla = """id,sede,fecha_retiro,dias_prestamo,estado_devolucion
1,Centro,2025-06-02,14,bueno
2,Norte,2025-06-02,7,regular
3,Centro,2025-06-03,21,bueno
4,Sur,2025-06-04,14,NULL
5,centro,2025-06-05,10,bueno
6,Norte,2025-06-05,999,regular
7,Centro,2025-06-06,14,dañado
8,Sur,2025-06-09,7,bueno
10,Centro,2025-06-10,28,NULL
11,NORTE,2025-06-11,14,bueno
12,Centro,11/06/2025,7,regular
13,Sur,2025-06-12,14,bueno
14,Centro ,2025-06-12,3,bueno
15,Norte,2025-06-13,21,dañado
16,Centro,2025-06-16,14,regular
18,Sur,2025-06-17,7,bueno
19,Centro,2025-06-17,,regular
20,Norte,2025-06-18,14,bueno
21,centro,18/06/2025,10,NULL
22,Sur,2025-06-19,45,bueno
23,Centro,2025-06-20,14,regular
24,Norte,2025-06-23,7,bueno
25,Sur,2025-06-24,21,regular
26,Centro,2025-06-24,14,bueno
26,Centro,2025-06-24,14,bueno
"""

Path("crudo/prestamos.csv").write_text(tabla, encoding="utf-8")

print("escrito: crudo/prestamos.csv")


---
# El pipeline empieza acá, leyendo el crudo

La celda de arriba fabrica el archivo para que este ejemplo funcione sin internet. **En tu
trabajo esa celda no existe:** ahí el crudo se descarga desde su URL, una sola vez, con este
patrón que ya conserva el archivo si está en disco:

```python
import urllib.request
if not Path("crudo/dato_crudo.csv").exists():
    urllib.request.urlretrieve(URL_DEL_ORGANISMO, "crudo/dato_crudo.csv")
```

Desde acá, todo lo demás es igual en el ejemplo y en tu trabajo. La regla de siempre: `crudo/` no
se toca más, y todo lo que el notebook produce va a `generado/`.


In [ ]:
df_crudo = pd.read_csv("crudo/prestamos.csv")

print("Filas y columnas:", df_crudo.shape)
df_crudo


## La primera mirada: los seis pasos de la clase 2

Todavía no se arregla nada. Primero el diagnóstico completo, en el orden de siempre: tamaño,
tipos, resumen, faltantes, duplicados y cardinalidad.


In [ ]:
print("1. TAMAÑO:", df_crudo.shape)
print()
print("2. TIPOS")
print(df_crudo.dtypes)
print()
print("3. RESUMEN de la única numérica de verdad")
print(df_crudo["dias_prestamo"].describe())


In [ ]:
print("4. FALTANTES por columna")
print(df_crudo.isna().sum())
print()
print("5. DUPLICADOS de fila completa:", df_crudo.duplicated().sum())
print()
print("6. CARDINALIDAD de las columnas de texto")
print("--- sede:", df_crudo["sede"].nunique(), "valores distintos")
print(df_crudo["sede"].value_counts().to_string())
print()
print("--- estado_devolucion:", df_crudo["estado_devolucion"].nunique(), "valores distintos")
print(df_crudo["estado_devolucion"].value_counts().to_string())


El perfilado ya encontró casi todo, y el informe de problemas se empieza a llenar acá:

- `dias_prestamo` tiene un máximo de **999**, que no es un préstamo de tres años: es un
  centinela, como el de la clase 2.
- Hay **una fila duplicada** exacta.
- `sede` tiene **seis valores distintos para tres sedes**: mayúsculas, minúsculas y un espacio
  invisible.
- Faltan **un** valor en `dias_prestamo` y **tres** en `estado_devolucion`.
- `fecha_retiro` e `id` son hallazgos que el perfilado no muestra solo: las fechas son texto con
  dos formatos mezclados, y la numeración de `id` salta el 9 y el 17. Dos préstamos se anularon o
  se perdieron, y eso es información sobre el instrumento.

Y un detalle que ya conocés de la clase 2: los tres faltantes de `estado_devolucion` dicen
`NULL` en el archivo, y pandas los convirtió solo. La celda que sigue lo comprueba releyendo el
crudo sin interpretar.


In [ ]:
sin_interpretar = pd.read_csv("crudo/prestamos.csv", keep_default_na=False)

print(sin_interpretar["estado_devolucion"].value_counts().to_string())


---
# El diccionario de variables

Se escribe antes de limpiar, porque la escala decide qué se puede calcular con cada columna, qué
gráfico le corresponde y cómo se codifica. El tipo lo dice pandas; la escala y el resto los pone
el que conoce el dato.

| variable | tipo | escala | descripción | valores admitidos |
|---|---|---|---|---|
| `id` | entero | identificador | Número del préstamo. Es una etiqueta: no se suma ni se promedia. | 1 en adelante, sin repetir |
| `sede` | texto | nominal | Sede donde se retiró el libro. | centro, norte, sur |
| `fecha_retiro` | texto | intervalo | Día en que se retiró el libro, en ISO 8601. | junio de 2025 |
| `dias_prestamo` | decimal | razón | Duración acordada del préstamo, en días. | 3 a 45 |
| `estado_devolucion` | texto | ordinal | Estado del libro al devolverlo, declarado por el bibliotecario. | dañado < regular < bueno |

Dos de esas celdas son decisiones y no hechos, y quedan declaradas: el rango de 3 a 45 días
supone que existen los préstamos especiales de vacaciones, y el orden de `estado_devolucion` lo
pusimos nosotros, porque el archivo no lo trae.


---
# La limpieza, en el orden de la clase 3

Unificar el texto, sacar lo que sobra, convertir los tipos, decidir sobre lo que falta, y recién
al final los extremos. Todo se hace sobre una copia, `df_limpio`, y el crudo queda intacto.
Después de cada paso, se mira el resultado: una limpieza que no se mira no está hecha.


In [ ]:
df_limpio = df_crudo.copy()

df_limpio["sede"] = df_limpio["sede"].str.strip()
df_limpio["sede"] = df_limpio["sede"].str.lower()

print("sede, de", df_crudo["sede"].nunique(), "formas a", df_limpio["sede"].nunique(), ":")
print(df_limpio["sede"].value_counts().to_string())


**Decisión 1, texto.** Seis formas pasan a tres con los dos pasos mecánicos: espacios de los
bordes y minúscula. Acá no hubo acentos que sacar; en la clase 3, sección 5.1, está el paso que
falta y el cuarto paso, el que decide sobre abreviaturas y no es mecánico. No se pierde nada que
el análisis use: la capitalización original no significaba nada.


In [ ]:
print("Duplicados de fila completa:", df_limpio.duplicated().sum())

df_limpio = df_limpio.drop_duplicates()

print("Filas: de", len(df_crudo), "a", len(df_limpio))


**Decisión 2, duplicados.** La última fila repetía entera la anterior, mismo `id` incluido: es
un error de carga y se saca. Veinticinco filas pasan a **veinticuatro préstamos**. El texto se
unificó antes, por la razón que la clase 3 mostró con números: un duplicado escrito distinto no
se reconoce. Lo que se pierde: si en esta biblioteca dos filas idénticas pudieran ser dos
préstamos reales del mismo libro el mismo día, estaríamos borrando uno. El diccionario dice que
`id` no se repite, así que el supuesto queda cubierto.


In [ ]:
fechas = pd.to_datetime(df_limpio["fecha_retiro"], format="ISO8601", errors="coerce")

sin_convertir = df_limpio.loc[fechas.isna(), "fecha_retiro"]
print("No estaban en ISO:", sin_convertir.tolist())

fechas = fechas.fillna(pd.to_datetime(sin_convertir, format="%d/%m/%Y"))
df_limpio["fecha_retiro"] = fechas

print("Convertidas:", df_limpio["fecha_retiro"].notna().sum(), "de", len(df_limpio))
print("Rango:", df_limpio["fecha_retiro"].min().date(), "a", df_limpio["fecha_retiro"].max().date())


**Decisión 3, fechas.** Dos pasos, como en la clase 3: primero lo que está en ISO, después las
dos rebeldes declarando su formato. El rango da del 2 al 24 de junio, que es el que la ficha
esperaba: esa comprobación es obligatoria, porque una conversión que no falla puede estar mal
igual. El supuesto que queda escrito: leímos `11/06/2025` como día/mes. Las filas vienen en
orden de retiro y esa fila cae entre el 11 y el 12 de junio, lo que confirma la lectura.


In [ ]:
df_limpio["dias_prestamo"] = df_limpio["dias_prestamo"].replace(999, float("nan"))

print("Faltantes por columna, después de convertir el centinela:")
print(df_limpio.isna().sum().to_string())


**Decisión 4, faltantes.** Tres casos distintos, tres decisiones distintas:

- El **999** era una ausencia disfrazada de dato, y promediarlo hubiera inflado todo: ahora es
  un faltante. También se puede declarar al leer, con `na_values`, como en la clase 3.
- El **vacío** de `dias_prestamo` y los tres **`NULL`** de `estado_devolucion` **se quedan como
  faltantes: ni se borran ni se imputan.** Borrar las filas tiraría las otras columnas, que
  están completas. E imputar inventaría un dato: un `NULL` acá puede ser un préstamo que todavía
  no se devolvió, y en ese caso no falta nada, la respuesta correcta es que no hay respuesta.
  Sin el diccionario del organismo no se puede distinguir, así que la decisión es no decidir y
  decirlo: todo análisis de `estado_devolucion` informa que se calcula sobre **veintiún**
  préstamos de veinticuatro.


In [ ]:
q1 = df_limpio["dias_prestamo"].quantile(0.25)
q3 = df_limpio["dias_prestamo"].quantile(0.75)
iqr = q3 - q1

extremos = df_limpio[df_limpio["dias_prestamo"] > q3 + 1.5 * iqr]

print("Q1:", q1, "| Q3:", q3, "| IQR:", iqr, "| límite superior:", q3 + 1.5 * iqr)
print()
print(extremos[["id", "sede", "fecha_retiro", "dias_prestamo"]].to_string(index=False))


**Decisión 5, extremos.** El IQR marca dos candidatos, el 28 y el 45. Un extremo no es un error:
se investiga cruzándolo con el resto de su fila y con lo que se sepa del dato. El 45 es
compatible con un préstamo especial de vacaciones, que el diccionario ya contemplaba, y el 28 es
una renovación normal. **Se conservan los dos y quedan anotados.** Sacarlos porque afean el
histograma sería elegir el resultado, que es la tercera forma de mentir de la clase 3.


## La codificación, lo último

Con el dato ya limpio. La consigna pide codificar **una** variable; acá van las dos de este
caso, para que se vea la diferencia. La escala del diccionario manda: la ordinal se codifica con
su orden, y a la nominal no se le inventa ninguno.


In [ ]:
orden = {"dañado": 1, "regular": 2, "bueno": 3}
df_limpio["estado_cod"] = df_limpio["estado_devolucion"].map(orden)

print("Con dato:", df_limpio["estado_cod"].notna().sum(), "de", len(df_limpio))
print("Mediana:", df_limpio["estado_cod"].median(), " <- 3, o sea 'bueno'")
print("El promedio no se calcula: las distancias entre categorías las inventamos nosotros.")

marcas_sede = pd.get_dummies(df_limpio["sede"], prefix="sede", dtype=int)
print()
print("Columnas nuevas para la nominal:", marcas_sede.columns.tolist())
print("Con la columna sucia hubieran salido seis, una por forma de escribir.")


---
# Dos gráficos, cada uno con su lectura

La escala elige el gráfico, como en la clase 3, sección 7.1. Y cada gráfico lleva su lectura
escrita: qué se ve y qué implica. Un gráfico sin lectura no cuenta.


In [ ]:
conteo = df_limpio["estado_devolucion"].value_counts()
conteo = conteo.reindex(["dañado", "regular", "bueno"])

conteo.plot.bar(title="Estado al devolver, en el orden de la escala")
plt.tight_layout()
plt.show()


**Lectura.** Barras en el orden de la escala, no por frecuencia, porque la variable es ordinal:
el `reindex` fuerza ese orden. La distribución carga hacia el buen estado, con dos devoluciones
dañadas sobre veintiuna con dato. La lectura informa el denominador: son veintiuno de
veinticuatro, porque tres préstamos no tienen estado registrado.


In [ ]:
df_limpio["dias_prestamo"].plot.hist(bins=10, title="Duración acordada del préstamo, en días")
plt.tight_layout()
plt.show()


**Lectura.** Histograma porque la duración es numérica de razón. El pico está en los 14 días,
que es el préstamo estándar, y la cola derecha son el 28 y el 45 que la limpieza decidió
conservar. Si los hubiéramos sacado en silencio, este gráfico mostraría una biblioteca donde los
préstamos largos no existen.


## Exportar el dataset limpio

El archivo va a `generado/`, con `index=False` porque el índice de
pandas no es un dato. El crudo queda como llegó: esa es toda la regla del taller en una línea de
`to_csv`.


In [ ]:
df_limpio.to_csv("generado/prestamos_limpio.csv", index=False)

print("Exportado: generado/prestamos_limpio.csv")
print("Crudo: ", df_crudo.shape, " <- intacto, releelo cuando quieras")
print("Limpio:", df_limpio.shape)


---
# El informe de problemas y decisiones

Se escribió mientras limpiábamos, y acá queda junto. Una fila por problema, con el número, la
dimensión de calidad de la clase 1, la decisión y su porqué. Esta tabla es el corazón del
trabajo real.

| # | Problema, con el número | Dimensión | Decisión | Justificación |
|---|---|---|---|---|
| 1 | `sede`: 6 formas de escribir 3 sedes | consistencia | strip + minúscula | Pasos mecánicos que no cambian el significado. Se pierde la capitalización, que no informaba nada. |
| 2 | 1 fila duplicada exacta sobre 25 | unicidad | borrar | Mismo `id` repetido entero: error de carga según el diccionario. Si fueran dos préstamos reales, se perdería uno: supuesto anotado. |
| 3 | 2 fechas en `dd/mm/aaaa` entre 22 en ISO | validez | convertir en dos pasos, formato declarado | El rango resultante, 2 al 24 de junio, coincide con la ficha. Lectura día/mes confirmada por el orden de las filas. |
| 4 | `dias_prestamo`: un 999 | validez | convertirlo en faltante | Es un centinela, no un préstamo: promediarlo inflaba la media de 14,5 a 57. |
| 5 | `estado_devolucion`: `NULL` en 3 filas de 24 | completitud | conservar como faltante | Puede ser "todavía no se devolvió", y ahí no falta nada. Sin el diccionario del organismo no se distingue: no se imputa y se informa el denominador, 21. |
| 6 | `dias_prestamo`: 28 y 45, marcados por IQR | exactitud | conservar y anotar | Compatibles con renovación y préstamo de vacaciones. Raro no es erróneo, y no se pudo investigar más. |

Y la cobertura de las siete dimensiones, incluidas las que dieron limpio: exactitud, validez,
completitud, consistencia y unicidad aparecen arriba. **Oportunidad:** el dato es de junio y se
usa en junio, sin problema. **Trazabilidad:** no hay diccionario del organismo ni se sabe quién
cargó cada fila; es el hallazgo que dejó dos decisiones sin poder cerrar del todo, y por eso se
reporta.


---
# Cierre

Esto fue un trabajo final entero, en miniatura: la fuente caracterizada, el diccionario escrito
antes de limpiar, seis problemas con números y decisiones justificadas, un pipeline que va del
crudo intacto al archivo exportado, y dos gráficos con su lectura.

Tu trabajo tiene la misma forma y otro tamaño. Las diferencias:

1. **El dato es real** y se descarga desde una URL: uno de los tres provistos en
   [`clase-4-trabajo-final/datos`](https://github.com/lautarodibartolo-ae/t8001-pre-procesado-de-datos/tree/main/clase-4-trabajo-final/datos), o uno que elijas de un portal.
2. **Los problemas no vienen plantados.** La consigna pide al menos cuatro; vas a encontrar más,
   y algunos no van a estar en ninguna lista.
3. **La herramienta la elegís vos.** Este ejemplo usa un notebook; la consigna dice qué cambia
   si trabajás en una planilla.

Lo que no cambia: el crudo intacto, el diccionario antes de la limpieza, cada decisión con su
porqué, y el que corrige pudiendo rehacer todo tu proceso.

La entrega es **opcional y recomendada**: es la única del taller, y la única forma de recibir una
devolución personal sobre tu criterio.

> **La consigna completa, con la guía de pasos:**
> [T8001_Consigna_Trabajo_Final.pdf](https://github.com/lautarodibartolo-ae/t8001-pre-procesado-de-datos/blob/main/clase-4-trabajo-final/T8001_Consigna_Trabajo_Final.pdf)
